# Introduction to Vector Stores


- Use Langchain documentation to create vector stores using chromadb, milvus, weaviate, pinecone: https://python.langchain.com/docs/integrations/vectorstores/
- Create embeddings of a document and index into vector stores: https://huggingface.co/blog/getting-started-with-embeddings
- For the task you will be indexing the book: https://www.planetebook.com/free-ebooks/crime-and-punishment.pdf
- Experiment different vector stores and other variables for and present your findings for optimizing retrieval (the most appropriate parts should be returned for any query)
- Hint: chunking

**Steps**
- Go through the given readings. Take an hour at max.
- Index document as vectors. Use any documentation to help you, but avoid using AI Tools.
- For embedding model, use an open source model from huggingface.
- Query should return most appropriate parts in relevance to the query. *Remember, your task is to build effective retrieval*


**Resources for understanding vector search**
- https://weaviate.io/blog/vector-search-explained


**Additional resources for understanding embeddings**
- https://cohere.com/llmu/text-embeddings
- https://docs.cohere.com/v2/docs/embeddings
- https://docs.cohere.com/v2/docs/playground-overview

##

## Required-Pakeges

In [1]:
## Required-Pakeges

!pip install -q langchain-community pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
from langchain_community.document_loaders import PyPDFLoader

/tmp/ipykernel_1487/4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### Dowanload Data

In [3]:
!wget -q https://www.planetebook.com/free-ebooks/crime-and-punishment.pdf -O crime_and_punishment.pdf

### Load_Pdf

In [4]:
loader = PyPDFLoader("crime_and_punishment.pdf")

documents = loader.load()

In [5]:
print("Total pages:", len(documents))

print("\nFirst page content:")
print(documents[0].page_content[:2000])

print("\nMetadata:")
print(documents[0].metadata)

Total pages: 767

First page content:
Download free eBooks of classic literature, books and 
novels at Planet eBook. Subscribe to our free eBooks blog 
and email newsletter.
Crime and Punishment
By Fyodor Dostoevsky

Metadata:
{'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0)', 'creationdate': '2008-02-06T20:52:29+11:00', 'subject': 'Download classic literature as completely free eBooks from Planet eBook.', 'author': 'Fyodor Dostoevsky', 'moddate': '2008-07-06T19:05:06+10:00', 'title': 'Crime and Punishment', 'trapped': '/False', 'source': 'crime_and_punishment.pdf', 'total_pages': 767, 'page': 0, 'page_label': '1'}


## Chunking

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [34]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

In [35]:
chunks = text_splitter.split_documents(documents)

In [36]:
print("Total pages:", len(documents))
print("Total chunks:", len(chunks))

Total pages: 767
Total chunks: 1501


In [37]:
# First 3 chunks ko check karte hain

for chunk in chunks[:3]:

    print("-----------------------------")
    print("Chunk:")
    print(chunk.page_content)

    print("Metadata:")
    print(chunk.metadata)

    print("Number of characters:")
    print(len(chunk.page_content))

-----------------------------
Chunk:
Download free eBooks of classic literature, books and 
novels at Planet eBook. Subscribe to our free eBooks blog 
and email newsletter.
Crime and Punishment
By Fyodor Dostoevsky
Metadata:
{'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0)', 'creationdate': '2008-02-06T20:52:29+11:00', 'subject': 'Download classic literature as completely free eBooks from Planet eBook.', 'author': 'Fyodor Dostoevsky', 'moddate': '2008-07-06T19:05:06+10:00', 'title': 'Crime and Punishment', 'trapped': '/False', 'source': 'crime_and_punishment.pdf', 'total_pages': 767, 'page': 0, 'page_label': '1'}
Number of characters:
177
-----------------------------
Chunk:
Crime and Punishment
Translator’s Preface
A 
few words about Dostoevsky himself may help the Eng -
lish reader to understand his work.
Dostoevsky was the son of a doctor. His parents were 
very hard- working and deeply religious people, but so poor 
that they lived with their five childre

## Embedding

In [38]:
# Install the required packages
!pip install -q sentence-transformers langchain-huggingface

In [39]:
# Import the Hugging Face embedding class
from langchain_huggingface import HuggingFaceEmbeddings

In [40]:
# Load the open-source embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [41]:
# Test the embedding model
test_text = "Crime and Punishment by Fyodor Dostoevsky"

test_embedding = embedding_model.embed_query(test_text)

print("Embedding dimensions:", len(test_embedding))
print("First 10 values:", test_embedding[:10])

Embedding dimensions: 384
First 10 values: [-0.06013427674770355, 0.05061093717813492, -0.1332448422908783, -0.016620533540844917, 0.04232260212302208, 0.09289436787366867, 0.09188763052225113, -0.007156626787036657, 0.012100864201784134, 0.05425141006708145]


#### Embeding for chunks

In [42]:
# Create embeddings for all the chunks

chunk_embeddings = []

for chunk in chunks:
    embedding = embedding_model.embed_query(chunk.page_content)
    chunk_embeddings.append(embedding)

### Check Embeding

In [43]:
# Check the embeddings

print("Total chunks:", len(chunks))
print("Total embeddings:", len(chunk_embeddings))

print("Embedding dimensions:", len(chunk_embeddings[0]))

Total chunks: 1501
Total embeddings: 1501
Embedding dimensions: 384


In [44]:
# Check the first chunk and its embedding

print("First chunk:")
print(chunks[0].page_content)

print("\nFirst chunk embedding (first 10 values):")
print(chunk_embeddings[0][:10])

First chunk:
Download free eBooks of classic literature, books and 
novels at Planet eBook. Subscribe to our free eBooks blog 
and email newsletter.
Crime and Punishment
By Fyodor Dostoevsky

First chunk embedding (first 10 values):
[0.0037111472338438034, -0.022727401927113533, -0.07402358949184418, -0.015953047201037407, 0.10638736933469772, 0.09371967613697052, -0.03373275697231293, 0.03340400010347366, 0.07733986526727676, 0.1109040305018425]


## Create Vector DB

In [45]:
!pip install -q --force-reinstall \
    "opentelemetry-api==1.43.0" \
    "opentelemetry-sdk==1.43.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-otlp-proto-grpc 1.44.0 requires opentelemetry-sdk~=1.44.0, but you have opentelemetry-sdk 1.43.0 which is incompatible.


In [46]:
# Import Chroma
from langchain_chroma import Chroma


# Create the vector store from our chunks and embedding model
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="crime_and_punishment"
)

In [47]:
print("Vector store created successfully!")
print("Number of documents:", vector_store._collection.count())

Vector store created successfully!
Number of documents: 4357


## Semilarity search

In [49]:
# Create a query
query = "Who is Raskolnikov?"

# Search for the most relevant chunks
results = vector_store.similarity_search(
    query,
    k=3
)

In [50]:
# Display the retrieved chunks

for result in results:
    print("-----------------------------")
    print(result.page_content)
    print("Metadata:", result.metadata)

-----------------------------
where Mr. Raskolnikov lived?’ He turned a rapid but atten -
tive look upon all three, especially upon Raskolnikov, to
Metadata: {'creator': 'Adobe InDesign CS2 (4.0)', 'subject': 'Download classic literature as completely free eBooks from Planet eBook.', 'total_pages': 767, 'moddate': '2008-07-06T19:05:06+10:00', 'creationdate': '2008-02-06T20:52:29+11:00', 'producer': 'Adobe PDF Library 7.0', 'source': 'crime_and_punishment.pdf', 'author': 'Fyodor Dostoevsky', 'page_label': '346', 'title': 'Crime and Punishment', 'page': 345, 'trapped': '/False'}
-----------------------------
seemed familiar to Raskolnikov came in.
Metadata: {'source': 'crime_and_punishment.pdf', 'title': 'Crime and Punishment', 'page': 191, 'total_pages': 767, 'trapped': '/False', 'moddate': '2008-07-06T19:05:06+10:00', 'page_label': '192', 'subject': 'Download classic literature as completely free eBooks from Planet eBook.', 'creator': 'Adobe InDesign CS2 (4.0)', 'author': 'Fyodor Dosto

## Get Similarity

In [51]:
# Search with similarity scores

results_with_scores = vector_store.similarity_search_with_score(
    query,
    k=3
)

In [52]:
# Display the results and their scores

for result, score in results_with_scores:

    print("-----------------------------")
    print("Score:", score)
    print("Chunk:")
    print(result.page_content)
    print("Metadata:", result.metadata)

-----------------------------
Score: 0.4579078257083893
Chunk:
where Mr. Raskolnikov lived?’ He turned a rapid but atten -
tive look upon all three, especially upon Raskolnikov, to
Metadata: {'subject': 'Download classic literature as completely free eBooks from Planet eBook.', 'creationdate': '2008-02-06T20:52:29+11:00', 'producer': 'Adobe PDF Library 7.0', 'total_pages': 767, 'page_label': '346', 'source': 'crime_and_punishment.pdf', 'author': 'Fyodor Dostoevsky', 'moddate': '2008-07-06T19:05:06+10:00', 'title': 'Crime and Punishment', 'trapped': '/False', 'creator': 'Adobe InDesign CS2 (4.0)', 'page': 345}
-----------------------------
Score: 0.4668648838996887
Chunk:
seemed familiar to Raskolnikov came in.
Metadata: {'page_label': '192', 'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0)', 'page': 191, 'source': 'crime_and_punishment.pdf', 'author': 'Fyodor Dostoevsky', 'trapped': '/False', 'title': 'Crime and Punishment', 'creationdate': '2008-02-06T20:52:29

# Retriever Experiemnet Multiple Querries

In [53]:
# Test questions

test_queries = [
    "Who is Raskolnikov?",
    "What crime did Raskolnikov commit?",
    "Who is Sonya?",
    "What happened to Marmeladov?",
    "Why did Raskolnikov kill the old woman?"
]

In [54]:
# Test each question

for query in test_queries:

    print("\nQUESTION:")
    print(query)

    results = vector_store.similarity_search_with_score(query, k=3)

    for result in results:

        print("\nScore:", result[1])
        print("Page:", result[0].metadata.get("page_label"))
        print("Text:")
        print(result[0].page_content[:500])


QUESTION:
Who is Raskolnikov?

Score: 0.4579078257083893
Page: 346
Text:
where Mr. Raskolnikov lived?’ He turned a rapid but atten -
tive look upon all three, especially upon Raskolnikov, to

Score: 0.4668648838996887
Page: 192
Text:
seemed familiar to Raskolnikov came in.

Score: 0.6012370586395264
Page: 681
Text:
1Free eBooks at Planet eBook.com
Intense disgust drew him away from Svidrigaïlov.
‘To think that I could for one instant have looked for help 
from that coarse brute, that depraved sensualist and black -
guard!’ he cried.
Raskolnikov’s judgment was uttered too lightly and 
hastily: there was something about Svidrigaïlov which 
gave him a certain original, even a mysterious character. 
As concerned his sister, Raskolnikov was convinced that

QUESTION:
What crime did Raskolnikov commit?

Score: 0.41176146268844604
Page: 746
Text:
Raskolnikov was not quite like an ordinary murderer and 
robber, but that there was another element in the case.
To the intense annoyance of those

#  Observation

### With **chunk size = 500** and **overlap = 50**, the retrieved results were often short, incomplete, and lacked sufficient context. After increasing them to **chunk size = 1000** and **overlap = 100**, the retrieved chunks became more complete and provided better contextual information, improving retrieval quality for several queries.
